# 🍾 Bottle Base Inspection — Binary Classifier
**Goal:** Predict whether a bottle is `GOOD (0)` or `FAULTY (1)` from an image of its base.

**Target:** F1 > 98% on the FAULTY class, with recall ≥ 99%.

---
## Architecture Overview
```
CSV (label + area_px)
  → Label Resolver       (3-tier logic → binary 0/1)
  → ROI Extractor        (Hough circle → crop → CLAHE → 224×224)
  → Augmentation         (Albumentations, train only)
  → EfficientNet-B0      (ImageNet pretrained backbone)
  → Classification Head  (Dropout → Linear → BN → ReLU → Linear(1))
  → Focal Loss           (γ=2, α=0.75 — hard-example focus)
  → Two-Phase Training   (freeze backbone → warm head → unfreeze → cosine LR)
  → Threshold Calibration(PR curve sweep on val, safety gate recall ≥ 0.99)
  → Evaluation           (F1, AUC, confusion matrix, plots)
```

## 0 — Install & Imports

In [ ]:
# ── Install missing packages (Kaggle already has torch/timm/albumentations) ──
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Uncomment if not present on your Kaggle instance
# pip_install('timm')
# pip_install('albumentations')
# pip_install('optuna')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# IMPORTS — all third-party dependencies in one place
# ═══════════════════════════════════════════════════════════════════════════

# Standard library
import os, math, json, logging, warnings, copy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Numerical / data
import numpy as np
import pandas as pd

# Image processing
import cv2
from PIL import Image

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Model zoo
import timm

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve,
)

# Hyperparameter search
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Visualisation
import matplotlib
matplotlib.use('Agg')   # safe for notebook + server both
import matplotlib.pyplot as plt
import seaborn as sns

# Misc
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

---
## 1 — Configuration

Everything in one dictionary. Change values here; no hunting through code.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MASTER CONFIG
# All hyperparameters live here. Modify this cell before running.
# ═══════════════════════════════════════════════════════════════════════════

CFG = {
    # ── Paths ────────────────────────────────────────────────────────────
    'annotation_csv':  '/kaggle/input/your-dataset/annotations.csv',
    'processed_dir':   '/kaggle/working/processed',   # pre-cropped images saved here
    'output_dir':      '/kaggle/working/outputs',
    'checkpoint_path': '/kaggle/working/best_model.pt',

    # ── Label columns in the CSV ─────────────────────────────────────────
    'label_col':  'label',
    'area_col':   'area_px',
    'image_col':  'image_path',

    # ── Image / ROI ───────────────────────────────────────────────────────
    'input_size':       224,
    'roi_method':       'hough',   # 'hough' | 'fixed'
    'fixed_cx':         1024,
    'fixed_cy':         1024,
    'fixed_radius':     900,
    'hough_dp':         1.2,
    'hough_min_dist':   500,
    'hough_param1':     100,
    'hough_param2':     40,
    'hough_min_r':      600,
    'hough_max_r':      1000,
    'roi_margin':       20,
    'clahe_clip':       2.0,
    'clahe_grid':       (8, 8),
    # ImageNet stats for EfficientNet pretrained weights
    'img_mean':         [0.485, 0.456, 0.406],
    'img_std':          [0.229, 0.224, 0.225],

    # ── Data splits ───────────────────────────────────────────────────────
    'train_frac': 0.70,
    'val_frac':   0.15,
    # test_frac = 1 - train_frac - val_frac = 0.15 (implicit)

    # ── Model ─────────────────────────────────────────────────────────────
    'backbone':    'efficientnet_b0',  # 'efficientnet_b0' | 'mobilenet_v3_small'
    'dropout1':    0.4,
    'hidden_dim':  256,
    'dropout2':    0.2,

    # ── Loss ─────────────────────────────────────────────────────────────
    'focal_gamma': 2.0,
    'focal_alpha': 0.75,

    # ── Training ─────────────────────────────────────────────────────────
    'total_epochs':    50,
    'phase1_epochs':   8,     # backbone frozen
    'phase2_epochs':   42,    # unfreeze last 3 blocks
    'warmup_epochs':   3,
    'batch_size':      64,
    'lr_phase1':       1e-3,
    'lr_phase2':       1e-4,
    'min_lr':          1e-7,
    'weight_decay':    1e-4,
    'grad_clip':       1.0,
    'amp':             True,    # mixed precision fp16
    'num_workers':     4,

    # ── Early stopping ────────────────────────────────────────────────────
    'early_stop_patience': 10,
    'early_stop_min_delta': 5e-4,

    # ── Threshold calibration ─────────────────────────────────────────────
    'min_recall_faulty': 0.990,   # safety constraint — never compromise below this
    'default_threshold': 0.5,

    # ── Optuna ────────────────────────────────────────────────────────────
    'optuna_n_trials':  30,
    'optuna_n_epochs':  12,   # short budget per trial
}

---
## 2 — Label Injection

### What it does
Converts a raw string label (e.g. `'air_bubble'`) plus an area measurement in pixels into a **binary integer 0 (GOOD) or 1 (FAULTY)**.

### The three-tier logic
| Tier | Rule | Example |
|------|------|---------|
| Always GOOD | Regardless of area | `water_drop`, `embossing` |
| Conditionally FAULTY | Only when `area_px > threshold` | `chip > 200px` → faulty |
| Always FAULTY | Regardless of area | `glass_shard`, `insect` |

### Innovation
Rather than a one-hot encoder that treats all labels equally, this enforces **domain expert knowledge** before any pixel is seen by the model. The area threshold approach handles borderline physical defects that are acceptable in small quantities but dangerous when large — this is industrial inspection domain logic, not ML convention.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LABEL DEFINITIONS
# Three lookup structures — the resolution logic reads them in order.
# ═══════════════════════════════════════════════════════════════════════════

# Tier 1: These labels are ALWAYS good regardless of area
ALWAYS_GOOD = frozenset([
    'embossing',
    'foam_residue',
    'no_fault',
    'water_drop',
])

# Tier 2: These labels are FAULTY only when area_px exceeds the threshold value
# Dict structure: { normalised_label_string : area_threshold_in_pixels }
CONDITIONAL_THRESHOLDS = {
    'air_bubble':            500,
    'chip':                  200,
    'contamination_light':   180,
    'glass_imperfection':    100,
    'scuffing':           75_000,
    'scuffing_heavy':      1_200,
}

# Tier 3: These labels are ALWAYS faulty regardless of area
ALWAYS_FAULTY = frozenset([
    'break_crack',
    'circlip',
    'contamination_dark',
    'crown_cap',
    'foil_semitransparent',
    'foreign_object_manual',
    'foreign_object_washing',
    'glass_shard',
    'insect',
    'label',
    'liquid',
    'mold',
    'no_base_visible',
    'paint_residue',
    'straw',
    'yeast_residue',
])


def normalise_label(raw: str) -> str:
    """
    Normalise a raw label string into a consistent key for lookup.
    - Strip whitespace
    - Lowercase
    - Replace spaces, hyphens, slashes with underscores

    Example: 'Break/Crack' -> 'break_crack'
    Example: 'Foreign Object - Manual Cleaning' -> 'foreign_object___manual_cleaning'
             then multi-underscore -> 'foreign_object_manual_cleaning'
    """
    normed = raw.strip().lower()
    for ch in (' ', '-', '/', '\\'):
        normed = normed.replace(ch, '_')
    # Collapse multiple consecutive underscores (e.g. from 'object - manual')
    while '__' in normed:
        normed = normed.replace('__', '_')
    return normed


def resolve_single_label(label: str, area_px: Optional[float] = None) -> int:
    """
    Convert one raw label + area into binary 0 (GOOD) or 1 (FAULTY).

    Parameters
    ----------
    label   : raw string from the annotation CSV (case-insensitive)
    area_px : defect area in pixels; only required for conditional labels.
              Pass None or NaN for unconditional labels.

    Returns
    -------
    int — 0 = GOOD (bottle is usable), 1 = FAULTY (bottle must be rejected)

    Design note
    -----------
    The function is intentionally conservative: any UNKNOWN label defaults to
    FAULTY. This is the safer industrial default — it is better to over-reject
    than to let a contaminated bottle pass.
    """
    key = normalise_label(label)

    # ── Tier 1: always good ───────────────────────────────────────────────
    if key in ALWAYS_GOOD:
        return 0

    # ── Tier 2: area-dependent ────────────────────────────────────────────
    if key in CONDITIONAL_THRESHOLDS:
        threshold = CONDITIONAL_THRESHOLDS[key]
        # If area is missing, assume worst case (conservative)
        if area_px is None or (isinstance(area_px, float) and math.isnan(area_px)):
            log.warning(
                "Label '%s' needs area_px but it is missing. Defaulting to FAULTY.",
                label,
            )
            return 1
        # Key comparison: strictly GREATER THAN threshold means faulty
        return int(float(area_px) > threshold)

    # ── Tier 3: always faulty ─────────────────────────────────────────────
    if key in ALWAYS_FAULTY:
        return 1

    # ── Unknown label — conservative default ─────────────────────────────
    log.warning("Unknown label '%s' (key='%s'). Defaulting to FAULTY.", label, key)
    return 1


def resolve_labels_dataframe(
    df: pd.DataFrame,
    label_col: str = 'label',
    area_col:  str = 'area_px',
) -> pd.DataFrame:
    """
    Apply resolve_single_label() to every row of the annotation DataFrame
    and add a 'binary_label' column.

    This is the ONLY function that should produce the 'binary_label' column.
    All downstream code reads from 'binary_label', never from the raw 'label'.

    Parameters
    ----------
    df        : DataFrame with at least a label column
    label_col : name of the raw string label column
    area_col  : name of the area column (may be absent for area-independent datasets)

    Returns
    -------
    Copy of df with an added 'binary_label' column (int 0/1)
    """
    df = df.copy()

    # Pull area series (or None for every row if the column doesn't exist)
    area_series = df[area_col] if area_col in df.columns else pd.Series([None] * len(df))

    df['binary_label'] = [
        resolve_single_label(lbl, area)
        for lbl, area in zip(df[label_col], area_series)
    ]

    # Log class distribution so you can spot imbalance immediately
    n_good   = (df['binary_label'] == 0).sum()
    n_faulty = (df['binary_label'] == 1).sum()
    total    = len(df)
    log.info(
        'Label resolution: GOOD=%d (%.1f%%) | FAULTY=%d (%.1f%%) | Total=%d',
        n_good,   100 * n_good   / total,
        n_faulty, 100 * n_faulty / total,
        total,
    )
    return df


# ── Quick sanity test ─────────────────────────────────────────────────────
def run_label_tests():
    TEST_CASES = [
        # (label,                area,    expected)
        ('no_fault',             None,    0),
        ('water_drop',           9000,    0),   # always good regardless of area
        ('embossing',            None,    0),
        ('foam_residue',         None,    0),
        ('air_bubble',           300,     0),   # 300 < 500 threshold → good
        ('air_bubble',           600,     1),   # 600 > 500 threshold → faulty
        ('chip',                 150,     0),   # 150 < 200 → good
        ('chip',                 250,     1),   # 250 > 200 → faulty
        ('contamination_light',  100,     0),   # exactly at threshold → good (not strictly greater)
        ('contamination_light',  181,     1),
        ('glass_imperfection',   50,      0),
        ('glass_imperfection',   120,     1),
        ('scuffing',             50_000,  0),
        ('scuffing',             80_000,  1),
        ('scuffing_heavy',       1200,    0),   # exactly at threshold → good
        ('scuffing_heavy',       1300,    1),
        ('glass_shard',          0,       1),   # always faulty
        ('insect',               None,    1),
        ('mold',                 None,    1),
        ('Break/Crack',          None,    1),   # normalisation test
        ('Foreign Object - Manual Cleaning', None, 1),  # complex normalisation
    ]
    passed = all_pass = True
    for label, area, expected in TEST_CASES:
        got = resolve_single_label(label, area)
        ok = (got == expected)
        if not ok:
            all_pass = False
            print(f'  FAIL  {label!r:40s}  area={str(area):>8}  got={got}  expected={expected}')
    print('All label tests PASSED ✓' if all_pass else 'SOME LABEL TESTS FAILED ✗')

run_label_tests()

---
## 3 — ROI Extraction

### What it does
Extracts the circular bottle base from a raw camera image by detecting the circle, masking everything outside it, and cropping to a tight square.

### Why each step matters
| Step | Why |
|------|-----|
| Median blur | More robust than Gaussian to the salt-and-pepper noise from industrial cameras |
| Hough Circle Transform | Finds the circular bottle base even when the conveyor belt or background is visible |
| Closest-to-centre selection | Handles false positives from reflections or background circles |
| Circular masking | Removes conveyor belt texture that would confuse the model |
| Square padding | Ensures consistent input shape without distorting the circle |
| CLAHE on LAB L-channel | Enhances local contrast specifically in the luminance channel, making contamination-light and glass imperfections dramatically more visible without over-saturating colours |

### Innovation
CLAHE in LAB space (not BGR) is the key technique here. Applying contrast enhancement to the L (lightness) channel only preserves colour information, which is critical for detecting `contamination_dark` vs `contamination_light`. Naive histogram equalisation on BGR would distort the colour signal.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ROI EXTRACTION — all functions operate on numpy arrays (BGR, as returned
# by cv2.imread) and return PIL Images ready for the augmentation pipeline.
# ═══════════════════════════════════════════════════════════════════════════

def detect_bottle_circle(
    gray_blurred: np.ndarray,
    cfg: dict,
    img_h: int,
    img_w: int,
) -> Tuple[int, int, int]:
    """
    Detect the circular bottle base using the Hough Circle Transform.
    Falls back to the fixed centre/radius from CFG if detection fails.

    How Hough Circles work
    ──────────────────────
    For every bright pixel in the edge map, the algorithm votes in a 3D
    accumulator (cx, cy, r). After all pixels have voted, peaks in the
    accumulator correspond to circles. param2 is the minimum vote count —
    lower values detect weaker circles but generate more false positives.

    Parameters
    ----------
    gray_blurred : grayscale, median-blurred image
    cfg          : CFG dictionary
    img_h, img_w : raw image dimensions for tie-breaking

    Returns
    -------
    (cx, cy, radius) as integers
    """
    circles = cv2.HoughCircles(
        gray_blurred,
        cv2.HOUGH_GRADIENT,
        dp=cfg['hough_dp'],
        minDist=cfg['hough_min_dist'],
        param1=cfg['hough_param1'],   # Canny upper edge threshold
        param2=cfg['hough_param2'],   # accumulator threshold — lower = more detections
        minRadius=cfg['hough_min_r'],
        maxRadius=cfg['hough_max_r'],
    )

    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        # If multiple circles detected, pick the one closest to the image centre.
        # Assumption: the bottle is always roughly centred (true for a fixed conveyor camera).
        cx_img, cy_img = img_w // 2, img_h // 2
        best = min(
            circles,
            key=lambda c: (c[0] - cx_img) ** 2 + (c[1] - cy_img) ** 2
        )
        return int(best[0]), int(best[1]), int(best[2])

    # Detection failed — use the fixed fallback from config
    log.warning('Hough detection failed, using fixed cx/cy/r fallback.')
    return cfg['fixed_cx'], cfg['fixed_cy'], cfg['fixed_radius']


def apply_clahe(bgr: np.ndarray, clip_limit: float, tile_grid: tuple) -> np.ndarray:
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalisation) to the
    luminance (L) channel in LAB colour space.

    Why LAB not BGR?
    ─────────────────
    LAB separates brightness (L) from colour (A, B). Applying CLAHE only to
    L enhances local contrast without shifting hues — critical for:
      - contamination_light: subtle brightness difference vs background
      - glass_imperfection: faint refractive patterns in the glass
      - scuffing: low-contrast scratches on clear glass

    clip_limit controls how aggressively contrast is boosted. Values 1-4 are
    typical for industrial inspection; >4 can amplify sensor noise.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)


def pad_to_square(img: np.ndarray) -> np.ndarray:
    """
    Pad the shorter axis of a rectangular crop with black pixels to make it
    a perfect square. Black padding is correct here because the circular mask
    already fills the outer ring with black — padding blends seamlessly.
    """
    h, w = img.shape[:2]
    if h == w:
        return img
    side = max(h, w)
    canvas = np.zeros((side, side, 3), dtype=img.dtype)
    y_off = (side - h) // 2
    x_off = (side - w) // 2
    canvas[y_off:y_off + h, x_off:x_off + w] = img
    return canvas


def extract_roi_from_array(bgr: np.ndarray, cfg: dict) -> Image.Image:
    """
    Full ROI pipeline for a single image (already loaded as BGR numpy array).

    Pipeline
    ────────
    1. Grayscale + median blur → Hough circle detection
    2. Circular mask (everything outside the bottle → black)
    3. Bounding square crop + margin
    4. Square padding if needed
    5. CLAHE contrast enhancement on L channel
    6. Resize to input_size x input_size (INTER_AREA = best for downsampling)
    7. Convert BGR → RGB → PIL Image

    Returns
    -------
    PIL.Image.Image  (RGB, input_size x input_size)
    """
    h, w = bgr.shape[:2]

    # Step 1 — detect circle
    if cfg['roi_method'] == 'fixed':
        cx, cy, radius = cfg['fixed_cx'], cfg['fixed_cy'], cfg['fixed_radius']
    else:
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        blurred = cv2.medianBlur(gray, 7)  # kernel=7: large enough to suppress noise, small enough to preserve circle edge
        cx, cy, radius = detect_bottle_circle(blurred, cfg, h, w)

    margin = cfg['roi_margin']

    # Step 2 — circular mask
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (cx, cy), radius + margin // 2, 255, thickness=-1)  # -1 = filled
    masked = cv2.bitwise_and(bgr, bgr, mask=mask)

    # Step 3 — bounding square crop
    x1 = max(cx - radius - margin, 0)
    y1 = max(cy - radius - margin, 0)
    x2 = min(cx + radius + margin, w)
    y2 = min(cy + radius + margin, h)
    crop = masked[y1:y2, x1:x2]

    # Step 4 — square padding
    crop = pad_to_square(crop)

    # Step 5 — CLAHE
    crop = apply_clahe(crop, cfg['clahe_clip'], cfg['clahe_grid'])

    # Step 6 — resize
    size = cfg['input_size']
    crop = cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)

    # Step 7 — BGR → RGB → PIL
    return Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))


def extract_roi_from_path(image_path: str, cfg: dict) -> Image.Image:
    """
    Convenience wrapper: load image from disk, then run extract_roi_from_array.
    """
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(f'Could not read: {image_path}')
    return extract_roi_from_array(bgr, cfg)


def preprocess_and_save_all(
    df: pd.DataFrame,
    output_dir: str,
    cfg: dict,
    image_col: str = 'image_path',
    n_jobs: int = 4,
) -> pd.DataFrame:
    """
    Pre-compute ROI crops for ALL images and save them to disk.
    Run this ONCE before training. Every training epoch then loads the fast
    pre-cropped image instead of re-running Hough detection.

    Returns a modified DataFrame where image_col now points to the
    pre-cropped files in output_dir.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    errors = []

    def _do_one(row):
        src = Path(row[image_col])
        dst = out / src.name
        if dst.exists():
            return str(dst), None
        try:
            pil = extract_roi_from_path(str(src), cfg)
            pil.save(str(dst))
            return str(dst), None
        except Exception as e:
            return str(dst), str(e)

    new_paths = []
    with ThreadPoolExecutor(max_workers=n_jobs) as pool:
        futures = {pool.submit(_do_one, row): i for i, row in df.iterrows()}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Pre-processing ROIs'):
            dst, err = fut.result()
            new_paths.append((futures[fut], dst, err))
            if err:
                errors.append(err)

    new_paths.sort(key=lambda x: x[0])  # restore original order
    df = df.copy()
    df[image_col] = [p for _, p, _ in new_paths]

    log.info('Pre-processing done. Errors: %d / %d', len(errors), len(df))
    return df


print('ROI extraction functions ready.')

---
## 4 — Augmentation Pipeline

### What it does
Applies randomised visual transformations **only during training** to make the model robust to real-world variation without needing to physically capture all those variations.

### Why each transform is chosen
| Transform | Physical justification |
|-----------|------------------------|
| ±180° rotation | Bottle bases are rotationally symmetric — there is no 'right way up' |
| H/V flip | Same symmetry reason |
| ColorJitter | Camera exposure fluctuations, lighting drift over the production day |
| GaussianBlur | Simulates lens defocus and camera vibration |
| GaussNoise | Industrial camera sensor noise, especially on older cameras |
| GridDistortion | Simulates subtle lens barrel/pincushion distortion across the image |
| CoarseDropout | Simulates small dust particles on the camera lens or minor occlusions |

### Innovation
Albumentations is used instead of torchvision transforms because it operates on numpy arrays in `uint8` format — the native format of OpenCV images — avoiding a wasteful float32 conversion during training. It is also 3–10× faster than PIL-based transforms for the same operations.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# AUGMENTATION PIPELINE
# ═══════════════════════════════════════════════════════════════════════════

def build_train_transforms(cfg: dict) -> A.Compose:
    """
    Build the TRAINING augmentation pipeline.

    Each transform is applied with a probability p. The order matters:
    geometric transforms come before photometric ones so that dropout
    holes don't get moved around, and normalisation is always last.

    The final two steps (Normalize + ToTensorV2) are non-random and always applied:
    - Normalize: applies ImageNet mean/std to match the pretrained backbone's
      expected input distribution. Without this, fine-tuning diverges.
    - ToTensorV2: converts HWC uint8 numpy array → CHW float32 torch.Tensor
    """
    size = cfg['input_size']
    return A.Compose([
        # ── Resize to model input (if not already done by ROI extractor) ──
        A.Resize(size, size),

        # ── Geometric ────────────────────────────────────────────────────
        # Full ±180° rotation: bottle bases have no orientation
        A.Rotate(limit=180, p=1.0),

        # Flip both axes — symmetric object
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        # Slight zoom variation simulates different bottle sizes / camera distances
        A.RandomResizedCrop(
            height=size, width=size,
            scale=(0.85, 1.0),   # crop between 85% and 100% of the image
            ratio=(0.95, 1.05),  # nearly square (bottle base is circular)
            p=0.5,
        ),

        # ── Photometric ───────────────────────────────────────────────────
        # Simulates lighting variation throughout the production day
        A.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02, p=0.7
        ),

        # Simulates lens defocus or camera vibration blur
        A.GaussianBlur(blur_limit=3, sigma_limit=(0.1, 1.5), p=0.3),

        # Industrial camera sensor noise
        A.GaussNoise(var_limit=(5.0, 25.0), p=0.25),

        # ── Structural ────────────────────────────────────────────────────
        # Lens barrel / pincushion distortion simulation
        A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),

        # Simulates dust on lens or minor image dropout
        # max_holes × max_height × max_width tells us the max occluded area
        A.CoarseDropout(max_holes=4, max_height=24, max_width=24, fill_value=0, p=0.3),

        # ── Always applied last ───────────────────────────────────────────
        # Normalise to ImageNet distribution expected by pretrained backbone
        A.Normalize(mean=cfg['img_mean'], std=cfg['img_std']),
        ToTensorV2(),  # (H,W,C) uint8 → (C,H,W) float32
    ])


def build_eval_transforms(cfg: dict) -> A.Compose:
    """
    Build the VALIDATION and TEST augmentation pipeline.
    No randomness — only resize and normalise.

    This is intentionally minimal so that validation metrics are deterministic
    and reproducible. Any randomness here would make val_f1 noisy and
    confuse the early-stopping logic.
    """
    size = cfg['input_size']
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=cfg['img_mean'], std=cfg['img_std']),
        ToTensorV2(),
    ])


print('Augmentation pipelines ready.')

---
## 5 — Dataset & DataLoader

### What it does
- `BottleDataset` maps a DataFrame row → (image tensor, label tensor) via the ROI extractor and augmentation pipeline
- `make_dataloaders` splits the data, builds a `WeightedRandomSampler` for the training set, and returns ready-to-use DataLoaders

### Class imbalance — two strategies working together
1. **`WeightedRandomSampler`**: each training epoch is drawn with replacement so that GOOD and FAULTY bottles appear approximately equally often. This prevents the model from learning 'always predict GOOD'
2. **Focal Loss** (Stage 6): even within a balanced batch, the loss down-weights easy examples, making the model focus on hard borderline cases

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════

class BottleDataset(Dataset):
    """
    PyTorch Dataset for bottle inspection.

    Kept as a class (not function) because PyTorch's DataLoader requires an
    object with __len__ and __getitem__. Everything else in this notebook is
    functional.

    Parameters
    ----------
    df         : DataFrame with columns [image_path, binary_label]
    transform  : Albumentations Compose pipeline
    cfg        : global CFG dict (for ROI extraction parameters)
    use_roi    : if True, run the full ROI extractor on each load;
                 if False, assume image_path already points to a pre-cropped image
    """
    def __init__(self, df: pd.DataFrame, transform, cfg: dict, use_roi: bool = False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.cfg       = cfg
        self.use_roi   = use_roi

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = row[self.cfg['image_col']]

        # Load image: either with full ROI pipeline or as plain PIL
        if self.use_roi:
            pil = extract_roi_from_path(path, self.cfg)
        else:
            pil = Image.open(path).convert('RGB')

        # Augment: Albumentations expects a uint8 numpy HWC array
        np_img = np.array(pil)
        augmented = self.transform(image=np_img)
        img_tensor = augmented['image']   # float32 [3, H, W]

        label_tensor = torch.tensor(row['binary_label'], dtype=torch.float32)
        return img_tensor, label_tensor


def compute_pos_weight(train_df: pd.DataFrame) -> torch.Tensor:
    """
    Compute the positive class weight for BCEWithLogitsLoss.

    pos_weight = n_negative / n_positive

    If there are 9x more GOOD bottles than FAULTY ones, pos_weight=9 tells the
    loss to treat each FAULTY prediction 9x more importantly than a GOOD one.
    This is used by weighted BCE; Focal Loss uses its own alpha instead.
    """
    n_good   = (train_df['binary_label'] == 0).sum()
    n_faulty = (train_df['binary_label'] == 1).sum()
    weight = torch.tensor([n_good / max(n_faulty, 1)], dtype=torch.float32)
    log.info('pos_weight=%.4f  (GOOD=%d, FAULTY=%d)', weight.item(), n_good, n_faulty)
    return weight


def make_weighted_sampler(labels: List[int]) -> WeightedRandomSampler:
    """
    Build a WeightedRandomSampler from a list of integer labels.

    Assigns each sample a weight inversely proportional to its class frequency.
    The sampler draws `len(labels)` samples with replacement each epoch, so
    rare classes appear just as often as common ones.

    Why with-replacement?
    ─────────────────────
    If FAULTY bottles are 1% of the dataset, without replacement we'd see
    very few of them per epoch. With replacement we oversample the minority
    class and undersample the majority, balancing exposure.
    """
    counts = np.bincount(labels)
    class_weights = 1.0 / counts   # inverse-frequency weights per class
    sample_weights = [float(class_weights[l]) for l in labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_dataloaders(
    df: pd.DataFrame,
    cfg: dict,
    use_roi: bool = False,
) -> Tuple[DataLoader, DataLoader, DataLoader, torch.Tensor]:
    """
    Split the annotated DataFrame and build train/val/test DataLoaders.

    Splitting strategy
    ──────────────────
    Stratified split ensures that the class ratio is preserved in every
    split. Without stratification, a small FAULTY class could end up
    entirely in training with zero test examples.

    Returns
    -------
    train_loader, val_loader, test_loader, pos_weight
    """
    # Step 1: stratified train / (val+test) split
    val_test_frac = 1.0 - cfg['train_frac']
    train_df, temp_df = train_test_split(
        df,
        test_size=val_test_frac,
        stratify=df['binary_label'],
        random_state=SEED,
    )

    # Step 2: stratified val / test split from the temp set
    val_frac_of_temp = cfg['val_frac'] / val_test_frac
    val_df, test_df = train_test_split(
        temp_df,
        test_size=1.0 - val_frac_of_temp,
        stratify=temp_df['binary_label'],
        random_state=SEED,
    )

    log.info('Split — train:%d  val:%d  test:%d', len(train_df), len(val_df), len(test_df))

    # Step 3: build datasets
    train_ds = BottleDataset(train_df, build_train_transforms(cfg), cfg, use_roi=use_roi)
    val_ds   = BottleDataset(val_df,   build_eval_transforms(cfg),  cfg, use_roi=use_roi)
    test_ds  = BottleDataset(test_df,  build_eval_transforms(cfg),  cfg, use_roi=use_roi)

    # Step 4: pos_weight for loss (computed from training set only)
    pos_w = compute_pos_weight(train_df)

    # Step 5: weighted sampler for training
    sampler = make_weighted_sampler(train_df['binary_label'].tolist())

    nw = cfg['num_workers']
    train_loader = DataLoader(
        train_ds, batch_size=cfg['batch_size'],
        sampler=sampler,   # sampler overrides shuffle
        num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg['batch_size'] * 2,
        shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg['batch_size'] * 2,
        shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
    )

    return train_loader, val_loader, test_loader, pos_w


print('Dataset and DataLoader functions ready.')

---
## 6 — Model Architecture

### What it does
Builds the complete neural network: a pretrained EfficientNet-B0 backbone plus a custom 2-layer classification head.

### Why EfficientNet-B0?
EfficientNet scales depth, width, and resolution simultaneously using a compound coefficient — empirically derived via neural architecture search. B0 is the base with ~5.3M parameters, giving an excellent accuracy/latency tradeoff. For this task it is large enough to model subtle defect patterns but small enough for sub-5ms inference with INT8 quantisation.

### Head design
```
GAP output [B, 1280]
  → Dropout(0.4)           ← regularise before the head
  → Linear(1280 → 256)     ← compress features
  → BatchNorm1d(256)        ← stabilise activations during fine-tuning
  → ReLU                   ← non-linearity
  → Dropout(0.2)            ← secondary regularisation
  → Linear(256 → 1)         ← scalar logit
  → (Sigmoid at inference)  ← P(FAULTY) in [0,1]
```

**Why no sigmoid during training?** `BCEWithLogitsLoss` and `BinaryFocalLoss` both combine the sigmoid and loss computation in a numerically stable way (log-sum-exp trick). Applying sigmoid separately and then computing log would lose precision for extreme logit values.

### Innovation: freeze/unfreeze API
Two functions `freeze_backbone` and `unfreeze_backbone_last_n` allow two-phase training without rewriting the training loop. Phase 1 trains only the head; Phase 2 unfreezes and fine-tunes together.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MODEL ARCHITECTURE
# ═══════════════════════════════════════════════════════════════════════════

# Feature dimension after Global Average Pooling for each backbone
BACKBONE_FEAT_DIMS = {
    'efficientnet_b0':   1280,
    'mobilenet_v3_small': 576,
    'resnet50':          2048,
}


def build_model(cfg: dict) -> nn.Module:
    """
    Construct and return the full BottleClassifier as an nn.Sequential-like model.

    The model is a standard nn.Module wrapping:
      1. A timm backbone (pretrained, head removed)
      2. A custom binary classification head

    timm's `num_classes=0` removes the original classification head and lets
    `global_pool='avg'` handle Global Average Pooling, outputting [B, feat_dim].

    Using timm (Py Torch Image Models) rather than torchvision gives access to
    300+ pretrained architectures with a consistent API, including newer
    EfficientNet variants trained on ImageNet-21k.
    """
    backbone_name = cfg['backbone']
    if backbone_name not in BACKBONE_FEAT_DIMS:
        raise ValueError(f'Unknown backbone: {backbone_name}. Choose from {list(BACKBONE_FEAT_DIMS)}')

    feat_dim = BACKBONE_FEAT_DIMS[backbone_name]

    # Load pretrained backbone (downloads weights on first call)
    backbone = timm.create_model(
        backbone_name,
        pretrained=True,
        num_classes=0,       # remove original classifier head
        global_pool='avg',   # Global Average Pooling → [B, feat_dim]
    )

    # Custom classification head
    head = nn.Sequential(
        nn.Dropout(p=cfg['dropout1']),
        nn.Linear(feat_dim, cfg['hidden_dim']),
        nn.BatchNorm1d(cfg['hidden_dim']),
        nn.ReLU(inplace=True),
        nn.Dropout(p=cfg['dropout2']),
        nn.Linear(cfg['hidden_dim'], 1),
        # No Sigmoid here — applied externally at inference time
    )

    # Kaiming uniform initialisation for Linear layers (better than default)
    for m in head.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    # Combine backbone + head into a simple module
    class BottleClassifier(nn.Module):
        def __init__(self, backbone, head):
            super().__init__()
            self.backbone = backbone
            self.head     = head

        def forward(self, x):
            return self.head(self.backbone(x))  # [B,1] logits

    model = BottleClassifier(backbone, head)
    log.info(
        'Model: %s | total params: %s | trainable: %s',
        backbone_name,
        f'{sum(p.numel() for p in model.parameters()):,}',
        f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}',
    )
    return model


def freeze_backbone(model: nn.Module) -> None:
    """
    Freeze ALL backbone parameters.
    Used at the start of Phase 1 so only the head is trained.

    Why freeze first?
    ─────────────────
    The head is initialised with random weights. If we immediately update
    the backbone with gradients from random head predictions, we destroy
    the carefully learned ImageNet features before the head has had any
    chance to learn the task. Freezing first lets the head stabilise.
    """
    for p in model.backbone.parameters():
        p.requires_grad = False
    log.info('Backbone frozen. Training head only.')


def unfreeze_backbone_last_n(model: nn.Module, n: int = 3) -> None:
    """
    Unfreeze the last `n` child modules of the backbone.

    Why only the last N blocks?
    ───────────────────────────
    Early layers of CNNs learn generic features (edges, textures) that transfer
    well from ImageNet. Later layers learn task-specific features. By only
    unfreezing the last N blocks, we fine-tune the high-level features while
    preserving the low-level ImageNet features. This reduces the risk of
    catastrophic forgetting with a small dataset.
    """
    children = list(model.backbone.children())
    total = len(children)
    for i, child in enumerate(children):
        requires = (i >= total - n)
        for p in child.parameters():
            p.requires_grad = requires
    # Always keep head trainable
    for p in model.head.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info('Last %d backbone blocks unfrozen. Trainable params: %s', n, f'{trainable:,}')


print('Model architecture functions ready.')

---
## 7 — Focal Loss

### What it does
A modified cross-entropy loss that automatically **reduces the weight of easy examples** and **increases the weight of hard ones**.

### The formula
```
FL(p_t) = -α_t · (1 - p_t)^γ · log(p_t)
```
- `p_t` = probability the model assigned to the correct class
- `(1 - p_t)^γ` = **focusing factor**: if the model is confident (p_t ≈ 1), this factor ≈ 0 → small loss. If the model is uncertain (p_t ≈ 0.5), this factor ≈ 0.25 → large loss.
- `α_t` = class-specific weight. Set higher for FAULTY to penalise missed defects more.

### Why not standard BCE?
With standard BCE, on a balanced batch the model can achieve a reasonable loss by correctly classifying the obvious easy cases (clear glass shard, perfectly clean bottle) and ignoring the hard borderline cases (contamination_light near the 180px threshold). Focal loss forces the model to keep learning from the hard cases even when easy ones are already well-classified.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════════════════════

def focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    gamma: float = 2.0,
    alpha: float = 0.75,
    reduction: str = 'mean',
) -> torch.Tensor:
    """
    Binary Focal Loss operating on raw logits (before sigmoid).

    Parameters
    ----------
    logits    : model output [B] or [B,1] — raw, before sigmoid
    targets   : binary float targets {0.0, 1.0} [B] or [B,1]
    gamma     : focusing parameter. 0 = standard BCE. 2 = original paper default.
    alpha     : weight for positive (FAULTY) class. Higher = more penalty for
                missing a faulty bottle. Typical range [0.5, 0.85].
    reduction : 'mean' (default) | 'sum' | 'none'

    Returns
    -------
    Scalar loss tensor

    Implementation note
    ───────────────────
    We compute BCE via F.binary_cross_entropy_with_logits which uses the
    log-sum-exp trick for numerical stability:
      BCE = max(x,0) - x*y + log(1 + exp(-|x|))
    This avoids log(0) issues when logits are extreme.
    """
    logits  = logits.view(-1)
    targets = targets.view(-1)

    # Per-sample BCE loss (no reduction)
    bce_per_sample = F.binary_cross_entropy_with_logits(
        logits, targets, reduction='none'
    )

    # p_t: probability the model assigned to the CORRECT class
    #   if target=1: p_t = sigmoid(logit)
    #   if target=0: p_t = 1 - sigmoid(logit)
    probs = torch.sigmoid(logits)
    p_t   = probs * targets + (1.0 - probs) * (1.0 - targets)

    # α_t: per-sample class weight
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)

    # Focal weight: (1 - p_t)^gamma
    #   High confidence (p_t ≈ 1) → weight ≈ 0 → easy example, almost no contribution
    #   Low confidence (p_t ≈ 0.5) → weight ≈ 0.25 → hard example, full contribution
    focal_weight = alpha_t * (1.0 - p_t) ** gamma

    loss = focal_weight * bce_per_sample

    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    return loss


print('Focal loss function ready.')

---
## 8 — Learning Rate Schedule

### What it does
Provides a two-phase schedule that matches the two-phase training strategy:
- **Phase 1** (epochs 0–7): Head warmup at constant high LR
- **Phase 2** (epochs 8–49): Fine-tuning starting from a lower LR, with cosine annealing that decays smoothly to near-zero

### Why cosine annealing?
Cosine annealing decays the LR smoothly (following the upper half of a cosine curve) rather than abruptly. This prevents the model from 'bouncing' around a minimum at the end of training. The cosine shape has become the standard for fine-tuning pretrained networks.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LEARNING RATE SCHEDULE
# ═══════════════════════════════════════════════════════════════════════════

def lr_lambda(epoch: int, cfg: dict) -> float:
    """
    Piecewise LR multiplier used with torch.optim.lr_scheduler.LambdaLR.

    The scheduler calls this function every epoch and multiplies the base LR
    (lr_phase1) by the returned float.

    Segments
    ────────
    0 … warmup_epochs:          Linear ramp 0 → 1.0  (prevents gradient explosion at start)
    warmup … phase1_end:        Constant 1.0          (stable head training)
    phase1_end … total_epochs:  Cosine decay from (lr_phase2/lr_phase1) → (min_lr/lr_phase1)

    Note: LambdaLR multiplies the *base LR* (lr_phase1) by this value, so
    phase-2 must return lr_phase2/lr_phase1 at its start.
    """
    w  = cfg['warmup_epochs']
    p1 = cfg['phase1_epochs']
    T  = cfg['total_epochs']
    lr1     = cfg['lr_phase1']
    lr2     = cfg['lr_phase2']
    min_lr  = cfg['min_lr']

    if epoch < w:
        # Linear warmup: epoch 0 → near-zero, epoch w → lr_phase1
        return (epoch + 1) / max(w, 1)

    if epoch < p1:
        # Constant phase-1 LR
        return 1.0

    # Cosine annealing for phase 2
    # Progress goes from 0 (epoch=p1) to 1 (epoch=T)
    p2_length = max(T - p1, 1)
    progress  = (epoch - p1) / p2_length
    cosine    = 0.5 * (1.0 + math.cos(math.pi * progress))
    # Interpolate between lr2 and min_lr
    target_lr = min_lr + (lr2 - min_lr) * cosine
    # Return as a multiplier of the base LR (lr_phase1)
    return target_lr / lr1


def build_optimizer_and_scheduler(model: nn.Module, cfg: dict):
    """
    Build AdamW optimizer and LambdaLR scheduler.

    Why AdamW?
    ──────────
    AdamW decouples weight decay from the gradient update (unlike vanilla
    Adam which conflates them). This gives cleaner regularisation and is
    now the standard for fine-tuning transformers and CNNs alike.
    """
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['lr_phase1'],
        weight_decay=cfg['weight_decay'],
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: lr_lambda(epoch, cfg),
    )

    return optimizer, scheduler


print('LR schedule functions ready.')

---
## 9 — Training Engine

### What it does
- `train_one_epoch`: one full pass over the training DataLoader, accumulating loss and updating weights
- `evaluate`: one pass over a DataLoader with no gradients, collecting predictions
- `train`: orchestrates the full training loop with phase switching, early stopping, and checkpointing

### AMP (Automatic Mixed Precision)
Training with fp16 (half precision) instead of fp32 for the forward pass and loss:
- ~2× faster on modern NVIDIA GPUs (tensor cores are optimised for fp16)
- ~2× less GPU memory (larger batch sizes)
- The `GradScaler` prevents fp16 underflow by scaling the loss before backprop and unscaling the gradients before the weight update

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAINING ENGINE
# ═══════════════════════════════════════════════════════════════════════════

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    cfg: dict,
    device: torch.device,
) -> float:
    """
    One training epoch.

    For each batch:
    1. Move data to device (non-blocking for async transfer)
    2. Forward pass under autocast (fp16 ops where safe)
    3. Compute focal loss
    4. GradScaler.scale(loss).backward() — scaled backprop
    5. Unscale + clip gradients (prevents gradient explosion)
    6. GradScaler.step(optimizer) — update weights
    7. GradScaler.update() — adjust scale factor for next step

    Returns
    -------
    Average loss over the epoch
    """
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # set_to_none=True is faster than zero_grad()

        # AMP forward pass: operations run in fp16 where numerically safe
        with torch.amp.autocast(device_type='cuda'):
            logits = model(images).squeeze(1)  # [B]
            loss   = focal_loss(logits, labels, cfg['focal_gamma'], cfg['focal_alpha'])

        # Scaled backprop (fp16-safe gradient computation)
        scaler.scale(loss).backward()

        # Unscale gradients before clipping (clipping operates on real gradient magnitudes)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    cfg: dict,
    threshold: float = 0.5,
) -> Dict[str, float]:
    """
    Run inference on a DataLoader and return all metrics.

    @torch.no_grad() disables gradient tracking, reducing memory and
    speeding up inference by ~20%.

    Returns
    -------
    dict with keys: loss, f1, precision, recall, accuracy, roc_auc,
                    pr_auc, probs (numpy array), labels (numpy array)
    """
    model.eval()
    all_probs:  List[float] = []
    all_labels: List[int]   = []
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(device_type='cuda'):
            logits = model(images).squeeze(1)
            loss   = focal_loss(logits, labels, cfg['focal_gamma'], cfg['focal_alpha'])

        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        total_loss += loss.item()

    probs_arr  = np.array(all_probs,  dtype=np.float32)
    labels_arr = np.array(all_labels, dtype=np.int32)
    preds_arr  = (probs_arr >= threshold).astype(int)

    try:
        roc_auc = roc_auc_score(labels_arr, probs_arr)
        pr_auc  = average_precision_score(labels_arr, probs_arr)
    except Exception:
        roc_auc = pr_auc = 0.0

    return {
        'loss':      total_loss / max(len(loader), 1),
        'f1':        f1_score(labels_arr, preds_arr, zero_division=0),
        'precision': precision_score(labels_arr, preds_arr, zero_division=0),
        'recall':    recall_score(labels_arr, preds_arr, zero_division=0),
        'accuracy':  accuracy_score(labels_arr, preds_arr),
        'roc_auc':   roc_auc,
        'pr_auc':    pr_auc,
        'probs':     probs_arr,
        'labels':    labels_arr,
    }


def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: dict,
    device: torch.device,
) -> Dict:
    """
    Full training loop with:
    - Two-phase backbone freezing
    - LR schedule
    - AMP (mixed precision)
    - Early stopping
    - Checkpoint saving (best val_f1)

    Returns
    -------
    history dict: {'train_loss', 'val_loss', 'val_f1', 'val_recall', 'lr'}
    """
    model = model.to(device)

    # Phase 1: freeze backbone, train head only
    freeze_backbone(model)
    optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)
    scaler = torch.amp.GradScaler(device_type='cuda')

    os.makedirs(Path(cfg['checkpoint_path']).parent, exist_ok=True)
    os.makedirs(cfg['output_dir'], exist_ok=True)

    best_f1       = 0.0
    no_improve    = 0
    history       = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_recall': [], 'lr': []}
    phase_switched = False

    for epoch in range(cfg['total_epochs']):

        # ── Phase switch ──────────────────────────────────────────────────
        if epoch == cfg['phase1_epochs'] and not phase_switched:
            log.info('=== Switching to Phase 2: unfreezing last 3 backbone blocks ===')
            unfreeze_backbone_last_n(model, n=3)
            # Rebuild optimizer with all newly unfrozen parameters
            optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)
            scaler = torch.amp.GradScaler(device_type='cuda')
            phase_switched = True

        # ── Train ─────────────────────────────────────────────────────────
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, cfg, device)
        scheduler.step()

        # ── Validate ──────────────────────────────────────────────────────
        val_metrics = evaluate(model, val_loader, device, cfg, threshold=cfg['default_threshold'])

        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1'].append(val_metrics['f1'])
        history['val_recall'].append(val_metrics['recall'])
        history['lr'].append(current_lr)

        log.info(
            'Epoch %3d | phase=%d | train_loss=%.4f | val_loss=%.4f | '
            'val_f1=%.4f | val_recall=%.4f | lr=%.2e',
            epoch + 1,
            1 if epoch < cfg['phase1_epochs'] else 2,
            train_loss,
            val_metrics['loss'],
            val_metrics['f1'],
            val_metrics['recall'],
            current_lr,
        )

        # ── Checkpoint: save if best val_f1 ──────────────────────────────
        if val_metrics['f1'] > best_f1 + cfg['early_stop_min_delta']:
            best_f1    = val_metrics['f1']
            no_improve = 0
            torch.save({
                'epoch':      epoch + 1,
                'state_dict': model.state_dict(),
                'val_f1':     best_f1,
                'cfg':        cfg,
            }, cfg['checkpoint_path'])
            log.info('  ↑ New best val_f1=%.4f — checkpoint saved.', best_f1)
        else:
            no_improve += 1

        # ── Early stopping ────────────────────────────────────────────────
        if no_improve >= cfg['early_stop_patience']:
            log.info('Early stopping at epoch %d (no improvement for %d epochs).', epoch + 1, no_improve)
            break

    log.info('Training complete. Best val_f1=%.4f', best_f1)
    return history


print('Training engine functions ready.')

---
## 10 — Hyperparameter Tuning (Optuna)

### What it does
Runs `n_trials` independent training runs with different hyperparameter combinations, guided by the **Tree-structured Parzen Estimator (TPE)** algorithm. TPE is a Bayesian optimisation method that builds a probabilistic model of which hyperparameters produce good results, and samples more from promising regions.

### Safety constraint
Any trial where `val_recall_faulty < 0.99` scores 0.0. This prevents Optuna from finding configurations that achieve high F1 by sacrificing recall (i.e. passing faulty bottles in exchange for fewer false rejections). In industrial quality control, a passed faulty bottle is far more costly than a rejected good bottle.

### When to run this
Run Optuna **before** full training with a short budget per trial (12–15 epochs). Then update CFG with the best parameters and run the full training (Stage 9).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HYPERPARAMETER TUNING
# ═══════════════════════════════════════════════════════════════════════════

def optuna_objective(
    trial: optuna.Trial,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    base_cfg: dict,
    n_epochs: int = 12,
) -> float:
    """
    Single Optuna trial.

    Samples hyperparameters, trains a model for `n_epochs`, and returns
    val_f1 as the objective to MAXIMISE.

    The safety constraint (recall ≥ 0.99) returns 0.0 immediately,
    which tells Optuna this is a bad region of parameter space to avoid.
    """
    cfg = copy.deepcopy(base_cfg)

    # ── Sample hyperparameters ────────────────────────────────────────────
    cfg['backbone']     = trial.suggest_categorical('backbone',    ['efficientnet_b0', 'mobilenet_v3_small'])
    cfg['lr_phase1']    = trial.suggest_float('lr_phase1',   1e-4,  5e-3, log=True)
    cfg['lr_phase2']    = trial.suggest_float('lr_phase2',   1e-6,  5e-4, log=True)
    cfg['focal_gamma']  = trial.suggest_float('focal_gamma', 1.0,   4.0)
    cfg['focal_alpha']  = trial.suggest_float('focal_alpha', 0.5,   0.9)
    cfg['dropout1']     = trial.suggest_float('dropout1',    0.2,   0.6)
    cfg['dropout2']     = trial.suggest_float('dropout2',    0.1,   0.4)
    cfg['hidden_dim']   = trial.suggest_categorical('hidden_dim',   [128, 256, 512])
    cfg['weight_decay'] = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    cfg['batch_size']   = trial.suggest_categorical('batch_size',   [32, 64, 128])

    # Short training budget for search
    cfg['total_epochs']  = n_epochs
    cfg['phase1_epochs'] = max(2, int(n_epochs * 0.2))  # 20% warmup
    cfg['early_stop_patience'] = 9999  # disable early stopping during search

    # ── Train ─────────────────────────────────────────────────────────────
    model = build_model(cfg).to(DEVICE)
    freeze_backbone(model)
    optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)
    scaler = torch.amp.GradScaler(device='cuda')

    best_f1 = 0.0
    for epoch in range(n_epochs):
        if epoch == cfg['phase1_epochs']:
            unfreeze_backbone_last_n(model, n=3)
            optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)

        train_one_epoch(model, train_loader, optimizer, scaler, cfg, DEVICE)
        scheduler.step()

        val_m = evaluate(model, val_loader, DEVICE, cfg)
        f1_val = val_m['f1']
        recall = val_m['recall']

        # Report intermediate value so Optuna can prune bad trials early
        trial.report(f1_val, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        best_f1 = max(best_f1, f1_val)

    # ── Safety constraint ─────────────────────────────────────────────────
    if recall < cfg['min_recall_faulty']:
        return 0.0  # penalise configurations that sacrifice safety

    return best_f1


def run_hyperparameter_search(
    train_loader: DataLoader,
    val_loader:   DataLoader,
    cfg: dict,
) -> dict:
    """
    Run the full Optuna hyperparameter search and return the best config.

    TPE sampler: builds a Gaussian mixture model of good vs bad parameter
    regions, then samples from the 'good' model (Expected Improvement).

    Median pruner: kills trials whose intermediate F1 is below the median
    of completed trials at the same epoch, avoiding wasting compute.
    """
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )

    study.optimize(
        lambda trial: optuna_objective(
            trial, train_loader, val_loader, cfg, n_epochs=cfg['optuna_n_epochs']
        ),
        n_trials=cfg['optuna_n_trials'],
        show_progress_bar=True,
    )

    best = study.best_trial
    log.info('Best Optuna trial #%d — val_f1=%.4f', best.number, best.value)
    for k, v in best.params.items():
        log.info('  %-20s = %s', k, v)

    # Build optimised config by patching best params into base config
    best_cfg = copy.deepcopy(cfg)
    for k, v in best.params.items():
        if k in best_cfg:
            best_cfg[k] = v

    return best_cfg, study


print('Hyperparameter tuning functions ready.')

---
## 11 — Threshold Calibration

### What it does
After training, the model outputs `P(FAULTY)` in [0, 1]. The default threshold of 0.5 is almost never optimal. This function sweeps all candidate thresholds from the precision-recall curve and selects the one that:
1. **Must satisfy**: `recall_faulty ≥ 0.99` (safety gate — never compromise this)
2. **Then maximises**: F1 score

### Why calibrate on val, not test?
The threshold is a hyperparameter. If we calibrate on the test set, we are effectively training on it, which would give an optimistic and misleading final F1. Calibrating on the validation set and reporting test performance is the correct protocol.

### Innovation
The safety-constrained threshold search is non-standard. Most practitioners just pick the F1-maximising threshold. Adding the `min_recall_faulty` constraint formalises the industrial requirement that **missing a faulty bottle is worse than falsely rejecting a good one**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# THRESHOLD CALIBRATION
# ═══════════════════════════════════════════════════════════════════════════

def compute_metrics_at_threshold(
    labels: np.ndarray,
    probs:  np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    """
    Compute all binary classification metrics given a specific threshold.

    This is the single source of truth for metric computation.
    All evaluation functions call this rather than implementing metrics inline.
    """
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    prec_faulty = tp / max(tp + fp, 1)
    rec_faulty  = tp / max(tp + fn, 1)
    f1_faulty   = 2 * prec_faulty * rec_faulty / max(prec_faulty + rec_faulty, 1e-9)

    try:
        roc = roc_auc_score(labels, probs)
        pr  = average_precision_score(labels, probs)
    except Exception:
        roc = pr = 0.0

    return {
        'threshold':         threshold,
        'f1_faulty':         f1_faulty,
        'f1_macro':          f1_score(labels, preds, average='macro', zero_division=0),
        'precision_faulty':  prec_faulty,
        'recall_faulty':     rec_faulty,
        'accuracy':          accuracy_score(labels, preds),
        'roc_auc':           roc,
        'pr_auc':            pr,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }


def calibrate_threshold(
    val_probs:  np.ndarray,
    val_labels: np.ndarray,
    min_recall: float = 0.99,
) -> Tuple[float, Dict]:
    """
    Find the optimal decision threshold by sweeping the precision-recall curve.

    Algorithm
    ─────────
    1. Compute the precision-recall curve (sklearn gives one threshold per point)
    2. For each threshold:
       a. Compute recall for FAULTY class
       b. If recall < min_recall: SKIP (safety gate)
       c. Compute F1
       d. Track best F1 and corresponding threshold
    3. If no threshold passes the safety gate, relax to best available

    Returns
    -------
    (best_threshold, metrics_dict)
    """
    precision_arr, recall_arr, thresholds = precision_recall_curve(val_labels, val_probs)
    # precision_recall_curve returns arrays of length N+1 for precision/recall
    # but length N for thresholds — align them
    thresholds = np.append(thresholds, 1.0)

    best_threshold = 0.5
    best_f1        = 0.0
    best_metrics   = {}
    fallback_threshold = 0.5
    fallback_f1        = 0.0

    for tau, _prec, rec in zip(thresholds, precision_arr, recall_arr):
        preds  = (val_probs >= tau).astype(int)
        f1_val = f1_score(val_labels, preds, zero_division=0)

        # Track best regardless of safety (for fallback)
        if f1_val > fallback_f1:
            fallback_f1        = f1_val
            fallback_threshold = float(tau)

        # Only consider thresholds that satisfy recall constraint
        if rec < min_recall:
            continue

        if f1_val > best_f1:
            best_f1        = f1_val
            best_threshold = float(tau)

    # If nothing passed the safety gate, use unconstrained best
    if best_f1 == 0.0:
        log.warning(
            'No threshold satisfies recall >= %.2f. Using unconstrained best (τ=%.4f).',
            min_recall, fallback_threshold
        )
        best_threshold = fallback_threshold

    best_metrics = compute_metrics_at_threshold(val_labels, val_probs, best_threshold)
    log.info(
        'Calibrated threshold=%.4f | F1(FAULTY)=%.4f | Recall(FAULTY)=%.4f',
        best_threshold, best_metrics['f1_faulty'], best_metrics['recall_faulty'],
    )
    return best_threshold, best_metrics


print('Threshold calibration functions ready.')

---
## 12 — Evaluation & Visualisation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION & PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def print_evaluation_report(metrics: dict, split: str = 'Test') -> None:
    """Human-readable evaluation summary printed to stdout."""
    sep = '─' * 62
    target_f1 = 0.98
    status = '✓ PASSED' if metrics['f1_faulty'] >= target_f1 else '✗ BELOW TARGET'
    print(f'\n{sep}')
    print(f'  BOTTLE INSPECTION — {split.upper()} SET EVALUATION')
    print(f'{sep}')
    print(f'  Decision threshold   : {metrics["threshold"]:.4f}')
    print(f'{sep}')
    print(f'  F1 (FAULTY class)    : {metrics["f1_faulty"]:.4f}   ← primary KPI')
    print(f'  Recall    (FAULTY)   : {metrics["recall_faulty"]:.4f}   ← safety metric')
    print(f'  Precision (FAULTY)   : {metrics["precision_faulty"]:.4f}')
    print(f'  F1 (macro)           : {metrics["f1_macro"]:.4f}')
    print(f'  Accuracy             : {metrics["accuracy"]:.4f}')
    print(f'  ROC-AUC              : {metrics["roc_auc"]:.4f}')
    print(f'  PR-AUC               : {metrics["pr_auc"]:.4f}')
    print(f'{sep}')
    print(f'  Confusion matrix:')
    print(f'    TP={metrics["tp"]:6d}   FP={metrics["fp"]:6d}')
    print(f'    FN={metrics["fn"]:6d}   TN={metrics["tn"]:6d}')
    print(f'{sep}')
    print(f'  F1 ≥ {target_f1:.0%} target        : {status}')
    print(f'{sep}\n')


def plot_training_history(history: dict, output_dir: str) -> None:
    """Plot train/val loss and val F1/recall over epochs."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history['train_loss'], label='Train loss', color='steelblue')
    axes[0].plot(history['val_loss'],   label='Val loss',   color='firebrick')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['val_f1'], color='darkorange')
    axes[1].axhline(0.98, color='green', linestyle='--', label='Target F1=0.98')
    axes[1].set_title('Val F1 (FAULTY)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(history['lr'], color='purple')
    axes[2].set_title('Learning rate'); axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_dir, 'training_history.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Training history saved → {path}')


def plot_evaluation_charts(
    probs:  np.ndarray,
    labels: np.ndarray,
    threshold: float,
    output_dir: str,
) -> None:
    """Four evaluation charts saved as a single figure."""
    preds = (probs >= threshold).astype(int)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # ── 1. Confusion matrix ───────────────────────────────────────────────
    cm = confusion_matrix(labels, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
        xticklabels=['GOOD', 'FAULTY'], yticklabels=['GOOD', 'FAULTY'],
    )
    axes[0, 0].set_title('Confusion Matrix')
    axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('True')

    # ── 2. Precision-Recall curve ─────────────────────────────────────────
    prec_arr, rec_arr, thr_arr = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    axes[0, 1].plot(rec_arr, prec_arr, lw=2, color='darkorange', label=f'AP={ap:.4f}')
    axes[0, 1].axvline(
        x=recall_score(labels, preds, zero_division=0),
        color='red', linestyle='--', lw=1.2, label=f'τ={threshold:.3f}'
    )
    axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision')
    axes[0, 1].set_title('Precision-Recall Curve')
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    # ── 3. ROC curve ─────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    axes[1, 0].plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC={auc:.4f}')
    axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[1, 0].set_xlabel('FPR'); axes[1, 0].set_ylabel('TPR')
    axes[1, 0].set_title('ROC Curve')
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    # ── 4. Confidence histogram ───────────────────────────────────────────
    bins = np.linspace(0, 1, 50)
    axes[1, 1].hist(probs[labels == 0], bins=bins, alpha=0.6, color='steelblue',  label='GOOD')
    axes[1, 1].hist(probs[labels == 1], bins=bins, alpha=0.6, color='firebrick',  label='FAULTY')
    axes[1, 1].axvline(threshold, color='black', linestyle='--', lw=1.5, label=f'τ={threshold:.3f}')
    axes[1, 1].set_xlabel('P(FAULTY)'); axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Confidence Score Distribution')
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_dir, 'evaluation_charts.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Evaluation charts saved → {path}')


def export_predictions(
    test_loader: DataLoader,
    probs:  np.ndarray,
    labels: np.ndarray,
    threshold: float,
    output_dir: str,
) -> pd.DataFrame:
    """
    Export a CSV with one row per test image containing:
    - image path
    - P(FAULTY) probability
    - predicted class (0/1)
    - true class (0/1)
    - human-readable predicted/true labels
    - whether the prediction was correct
    """
    preds = (probs >= threshold).astype(int)
    df    = test_loader.dataset.df.copy()

    df['prob_faulty']    = probs
    df['pred_binary']    = preds
    df['true_binary']    = labels
    df['pred_class']     = ['FAULTY' if p == 1 else 'GOOD' for p in preds]
    df['true_class']     = ['FAULTY' if l == 1 else 'GOOD' for l in labels]
    df['correct']        = (preds == labels)
    df['threshold_used'] = threshold

    path = os.path.join(output_dir, 'predictions.csv')
    df.to_csv(path, index=False)
    print(f'Predictions exported → {path}  ({len(df)} rows)')
    return df


print('Evaluation and visualisation functions ready.')

---
## 13 — Main Execution

This cell ties everything together. Run it to go from raw CSV to a trained, calibrated, and evaluated model.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION PIPELINE
# ═══════════════════════════════════════════════════════════════════════════

def run_full_pipeline(cfg: dict = CFG, run_hparam_search: bool = False):
    """
    Full end-to-end pipeline:

    1.  Load annotation CSV
    2.  Resolve raw labels → binary 0/1
    3.  (Optional) Pre-process all images → ROI crops saved to disk
    4.  Build DataLoaders with stratified splits and weighted sampling
    5.  (Optional) Optuna hyperparameter search → update CFG
    6.  Build model
    7.  Train with two-phase strategy
    8.  Load best checkpoint
    9.  Calibrate threshold on validation set
    10. Final evaluation on held-out test set
    11. Save plots and predictions CSV
    """

    os.makedirs(cfg['output_dir'], exist_ok=True)

    # ── Step 1-2: Load and resolve labels ────────────────────────────────
    print('\n═══ Step 1: Loading and resolving labels ═══')
    df = pd.read_csv(cfg['annotation_csv'])
    df = resolve_labels_dataframe(df, label_col=cfg['label_col'], area_col=cfg['area_col'])

    # ── Step 3 (optional): Pre-process ROIs to disk ───────────────────────
    # RECOMMENDED: run this once, then set use_roi=False in DataLoaders.
    # If your images are already pre-cropped, skip this block.
    PREPROCESS_FIRST = False  # ← Set True on first run
    if PREPROCESS_FIRST:
        print('\n═══ Step 3: Pre-processing ROIs to disk ═══')
        df = preprocess_and_save_all(df, cfg['processed_dir'], cfg, n_jobs=4)
        # After this, df[image_col] points to pre-cropped images
        cfg['annotation_csv'] = os.path.join(cfg['processed_dir'], 'annotations_processed.csv')
        df.to_csv(cfg['annotation_csv'], index=False)

    # use_roi=True: run Hough detection on each load (slower but no pre-processing needed)
    # use_roi=False: images in df are already pre-cropped (faster)
    use_roi = not PREPROCESS_FIRST

    # ── Step 4: DataLoaders ───────────────────────────────────────────────
    print('\n═══ Step 4: Building DataLoaders ═══')
    train_loader, val_loader, test_loader, pos_weight = make_dataloaders(df, cfg, use_roi=use_roi)
    print(f'Batches — train:{len(train_loader)}  val:{len(val_loader)}  test:{len(test_loader)}')

    # ── Step 5 (optional): Hyperparameter search ──────────────────────────
    if run_hparam_search:
        print('\n═══ Step 5: Hyperparameter search (Optuna) ═══')
        cfg, study = run_hyperparameter_search(train_loader, val_loader, cfg)
        print('Best hyperparameters applied to CFG.')

    # ── Step 6: Build model ───────────────────────────────────────────────
    print('\n═══ Step 6: Building model ═══')
    model = build_model(cfg)

    # ── Step 7: Train ─────────────────────────────────────────────────────
    print('\n═══ Step 7: Training ═══')
    history = train(model, train_loader, val_loader, cfg, DEVICE)
    plot_training_history(history, cfg['output_dir'])

    # ── Step 8: Load best checkpoint ─────────────────────────────────────
    print('\n═══ Step 8: Loading best checkpoint ═══')
    ckpt = torch.load(cfg['checkpoint_path'], map_location=DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (val_f1={ckpt["val_f1"]:.4f})')
    model = model.to(DEVICE)

    # ── Step 9: Threshold calibration on val set ─────────────────────────
    print('\n═══ Step 9: Calibrating threshold on validation set ═══')
    val_metrics = evaluate(model, val_loader, DEVICE, cfg, threshold=0.5)
    threshold, val_calib_metrics = calibrate_threshold(
        val_metrics['probs'], val_metrics['labels'],
        min_recall=cfg['min_recall_faulty'],
    )
    print(f'Calibrated threshold: {threshold:.4f}')
    print_evaluation_report(val_calib_metrics, split='Validation (calibration)')

    # ── Step 10: Final evaluation on test set ─────────────────────────────
    print('\n═══ Step 10: Final evaluation on test set ═══')
    test_eval = evaluate(model, test_loader, DEVICE, cfg, threshold=threshold)
    test_metrics = compute_metrics_at_threshold(test_eval['labels'], test_eval['probs'], threshold)
    print_evaluation_report(test_metrics, split='Test')

    # sklearn detailed report
    preds = (test_eval['probs'] >= threshold).astype(int)
    print(classification_report(
        test_eval['labels'], preds,
        target_names=['GOOD', 'FAULTY'], digits=4,
    ))

    # ── Step 11: Save outputs ─────────────────────────────────────────────
    print('\n═══ Step 11: Saving evaluation outputs ═══')
    plot_evaluation_charts(
        test_eval['probs'], test_eval['labels'],
        threshold, cfg['output_dir'],
    )
    pred_df = export_predictions(
        test_loader, test_eval['probs'], test_eval['labels'],
        threshold, cfg['output_dir'],
    )
    # Save metrics JSON
    metrics_path = os.path.join(cfg['output_dir'], 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump({k: round(float(v), 6) for k, v in test_metrics.items()}, f, indent=2)
    print(f'Metrics JSON saved → {metrics_path}')

    print('\n═══ PIPELINE COMPLETE ═══')
    return model, threshold, test_metrics, pred_df


# ── RUN ───────────────────────────────────────────────────────────────────
# Set run_hparam_search=True to run Optuna first (recommended for first run)
model, threshold, metrics, predictions = run_full_pipeline(
    cfg=CFG,
    run_hparam_search=False,  # ← change to True to tune hyperparameters first
)

# save final model
final_model_path = os.path.join(CFG['output_dir'], 'final_model.pth')
torch.save(model.state_dict(), final_model_path)
print(f'Final model weights saved → {final_model_path}')

---
## 14 — Single Image Inference

Once the model is trained, use this function to predict on a single bottle image.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SINGLE IMAGE INFERENCE
# ═══════════════════════════════════════════════════════════════════════════

def predict_single_image(
    image_path: str,
    model: nn.Module,
    threshold: float,
    cfg: dict,
    device: torch.device,
    show: bool = True,
) -> dict:
    """
    Predict whether a single bottle image is GOOD or FAULTY.

    Runs the full pipeline:
    1. ROI extraction (Hough circle → crop → CLAHE → resize)
    2. Normalisation
    3. Model inference
    4. Threshold comparison

    Parameters
    ----------
    image_path : path to a raw bottle base image
    model      : trained BottleClassifier
    threshold  : calibrated decision threshold
    cfg        : global CFG
    device     : inference device
    show       : whether to display the processed image in the notebook

    Returns
    -------
    dict with keys: prob_faulty, prediction (0/1), label (GOOD/FAULTY), confident
    """
    model.eval()

    # Step 1: ROI extraction
    pil_img = extract_roi_from_path(image_path, cfg)

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(Image.open(image_path))
        axes[0].set_title('Raw image'); axes[0].axis('off')
        axes[1].imshow(pil_img)
        axes[1].set_title('ROI crop (input to model)'); axes[1].axis('off')
        plt.tight_layout(); plt.show()

    # Step 2: Normalise and convert to tensor
    transform = build_eval_transforms(cfg)
    np_img  = np.array(pil_img)
    tensor  = transform(image=np_img)['image'].unsqueeze(0).to(device)  # [1,3,H,W]

    # Step 3: Inference
    with torch.no_grad():
        with torch.amp.autocast(device_type = 'cuda'):
            logit = model(tensor).squeeze()  # scalar
    prob_faulty = float(torch.sigmoid(logit).cpu())

    # Step 4: Decision
    prediction = int(prob_faulty >= threshold)
    label      = 'FAULTY' if prediction == 1 else 'GOOD'

    # Confidence: how far from the threshold?
    confident  = abs(prob_faulty - threshold) > 0.2

    result = {
        'prob_faulty': round(prob_faulty, 4),
        'prediction':  prediction,
        'label':       label,
        'confident':   confident,
        'threshold':   threshold,
    }

    print(f'\n  Image      : {os.path.basename(image_path)}')
    print(f'  P(FAULTY)  : {prob_faulty:.4f}')
    print(f'  Decision   : {label} (threshold={threshold:.4f})')
    print(f'  Confident  : {"Yes" if confident else "Borderline"}')

    return result


# ── Example usage (replace with an actual image path from your dataset) ───
# result = predict_single_image(
#     image_path='/kaggle/input/your-dataset/bottle_001.png',
#     model=model,
#     threshold=threshold,
#     cfg=CFG,
#     device=DEVICE,
#     show=True,
# )
print('Single-image inference function ready.')

---
## Appendix — How the Multiple Classes Are Handled

This is a **binary classification model**, but it handles 25+ raw label classes through the three-tier system before training begins.

```
25+ raw labels
    │
    ├── Tier 1 (always GOOD) ─────────────────────────────────────────────→ 0
    │     embossing, foam_residue, no_fault, water_drop
    │
    ├── Tier 2 (area-dependent) ─── area > threshold ──────────────────────→ 1
    │   │         air_bubble>500  chip>200  contamination_light>180        ↑
    │   │         glass_imperfection>100  scuffing>75000  scuffing_heavy>1200
    │   └──────── area ≤ threshold ──────────────────────────────────────→ 0
    │
    └── Tier 3 (always FAULTY) ────────────────────────────────────────────→ 1
          break_crack, circlip, contamination_dark, crown_cap,
          foil_semitransparent, foreign_object_*, glass_shard, insect,
          label, liquid, mold, no_base_visible, paint_residue, straw,
          yeast_residue
                              │
                              ▼
                    binary_label {0, 1}
                              │
                              ▼
                   EfficientNet-B0 → sigmoid → P(FAULTY)
                              │
                              ▼
                  P(FAULTY) ≥ threshold → REJECT
                  P(FAULTY) <  threshold → ACCEPT
```

The conditional tier is the key innovation. `scuffing` with 50,000px is a GOOD bottle. `scuffing` with 80,000px is FAULTY. The model never sees this distinction as a multi-class problem — the domain expert threshold logic is applied before the model sees any pixel, and the model only needs to learn one decision boundary.